# 04 — LFM2.5-Audio-1.5B: Finetuning mit `liquid-audio`

Andere Baustelle als die anderen drei Notebooks: Audio läuft **nicht** über TRL,
sondern über Liquid's eigenes Paket
[`liquid-audio`](https://github.com/Liquid4All/liquid-audio) (Finetuning ab v1.2.0).

Ablauf:

1. Rohdaten auf `list[ChatMessage]` abbilden (`TextSegment`, `AudioSegment`,
   `InterleavedSegment`).
2. `LFM2AudioChatMapper` + `preprocess_dataset()` → vorverarbeitetes Dataset.
3. `LFM2DataLoader` + `liquid_audio.trainer.Trainer` → Training.

**Wichtig:** dieser Trainer macht ein **volles Finetuning**, er hat keinen
LoRA-Schalter. VRAM-Bedarf ist eine andere Liga als bei den QLoRA-Tracks —
**A100 / L4 einplanen**, die freie T4 wird eng.

In [ ]:
!pip install -q -U liquid-audio datasets soundfile
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/muscal-lfm/audio')
(DRIVE_DIR / 'data').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'outputs').mkdir(parents=True, exist_ok=True)
print("drive project dir:", DRIVE_DIR)
print("contents:", sorted(p.name for p in DRIVE_DIR.iterdir()))

## Aufgabe festlegen

Der System-Prompt ist bei diesem Modell **funktional**, nicht dekorativ — er
wählt den Modus. Die vier vom Modell gelernten Prompts:

| Aufgabe | System-Prompt |
|---|---|
| ASR | `Perform ASR.` |
| TTS | `Perform TTS. Use the US female voice.` (US/UK × male/female) |
| Voice Chat | `Respond with interleaved text and audio.` |

In [ ]:
SYSTEM_PROMPT = "Perform ASR."          # <- fuer deine Aufgabe anpassen
MODEL_ID      = "LiquidAI/LFM2.5-Audio-1.5B"
DATA_DIR      = DRIVE_DIR / "data"
PREPROC_DIR   = DRIVE_DIR / "data" / "audio" / "train"
OUT_DIR       = DRIVE_DIR / "outputs" / "lfm25-audio"

CONTEXT_LENGTH = 256
BATCH_SIZE     = 8
MAX_STEPS      = 1000
LR             = 1e-4

print(SYSTEM_PROMPT, "|", MODEL_ID)

## 1. Rohdaten → ChatMessages

Unten ein Iterator für ein ASR-Setup (`audio` + `transcription`). Für TTS wird
getauscht: `user` bekommt den `TextSegment`, `assistant` den `AudioSegment`.
Für Voice-Chat kommen `InterleavedSegment(text=..., audio=...)` in die
Assistenten-Nachricht.

`audio` muss **Bytes eines Audiocontainers** sein (wav/flac/ogg/…), keine
vorverarbeiteten Features — der Processor macht daraus log-mel.

In [ ]:
from pathlib import Path

# %%writefile interpoliert keine Variablen -- deshalb schreiben wir die Datei
# selbst, mit den oben gesetzten Werten eingesetzt.
TEMPLATE = '''
"""Angepasst von liquid-audio/examples/preprocess_jenny_tts.py"""
from __future__ import annotations

from collections.abc import Iterator

from datasets import Audio, load_dataset

from liquid_audio import LFM2AudioProcessor
from liquid_audio.data.mapper import LFM2AudioChatMapper
from liquid_audio.data.preprocess import preprocess_dataset
from liquid_audio.data.types import AudioSegment, ChatMessage, TextSegment

SYSTEM_PROMPT = "{system_prompt}"
MODEL_ID = "{model_id}"
OUTPUT_PATH = "{preproc_dir}"


class MyDataIterator:
    """Passt diese drei Zeilen an dein Dataset an."""

    def __init__(self, split: str = "train") -> None:
        self.ds = load_dataset("mein/dataset", split=split)
        self.ds = self.ds.cast_column("audio", Audio(decode=False))

    def __iter__(self) -> Iterator[list[ChatMessage]]:
        for row in self.ds:
            audio_bytes = row["audio"]["bytes"]        # Audio-Spalte
            transcript = row["transcription"]          # Text-Spalte
            yield [
                ChatMessage(role="system", content=[TextSegment(text=SYSTEM_PROMPT)]),
                ChatMessage(role="user", content=[AudioSegment(audio=audio_bytes)]),
                ChatMessage(role="assistant", content=[TextSegment(text=transcript)]),
            ]


if __name__ == "__main__":
    processor = LFM2AudioProcessor.from_pretrained(MODEL_ID, device="cuda").eval()
    mapper = LFM2AudioChatMapper(processor)
    preprocess_dataset(
        data=MyDataIterator(),
        output_path=OUTPUT_PATH,
        mapper=mapper,
        max_context_length=256,   # laengere Samples werden uebersprungen
    )
'''

Path("preprocess_my_data.py").write_text(
    TEMPLATE.format(
        system_prompt=SYSTEM_PROMPT,
        model_id=MODEL_ID,
        preproc_dir=str(PREPROC_DIR),
    ),
    encoding="utf-8",
)
print(Path("preprocess_my_data.py").read_text(encoding="utf-8")[:400])

Passe `MyDataIterator` an dein Dataset an (Spaltennamen, Rollenverteilung), dann
Preprocessing starten. Das schreibt das vorverarbeitete Dataset nach
`data/audio/train` — bei großen Sets dauert das, also einmal laufen lassen und in
Drive lassen.

In [ ]:
!python preprocess_my_data.py
!ls -la "$PREPROC_DIR" | head -20

## 2. Trainieren

`liquid_audio.trainer.Trainer` übernimmt Loop, Logging und Checkpoints
(`save_interval`, `val_interval`). Checkpoints landen in `output_dir` — auf Drive
zeigen lassen, damit ein Runtime-Reset nichts verliert.

In [ ]:
from pathlib import Path

from liquid_audio.data.dataloader import LFM2DataLoader
from liquid_audio.trainer import Trainer

assert Path(PREPROC_DIR).exists(), "erst preprocess_my_data.py laufen lassen"

train_data = LFM2DataLoader(dataset_path=str(PREPROC_DIR), context_length=CONTEXT_LENGTH)

trainer = Trainer(
    model_id=MODEL_ID,
    train_data=train_data,
    lr=LR,
    batch_size=BATCH_SIZE,
    max_steps=MAX_STEPS,
    warmup_steps=50,
    dataloader_num_workers=2,
    logging_interval=10,
    save_interval=250,
    val_interval=100,
    output_dir=str(OUT_DIR),
)
trainer.train()

## 3. Testen

Der Processor wandelt Tokens zurück in Waveform (24 kHz). Bei ASR ist die Ausgabe
capitalized und interpungiert — genau so, wie es im Training stand.

In [ ]:
import torch, soundfile as sf
from liquid_audio import LFM2AudioModel, LFM2AudioProcessor, ChatState

processor = LFM2AudioProcessor.from_pretrained(MODEL_ID).eval()
model = LFM2AudioModel.from_pretrained(MODEL_ID).eval().cuda()

wav, sr = sf.read("mein_test_audio.wav", dtype="float32")
wav = torch.from_numpy(wav).unsqueeze(0)

chat = ChatState(processor)
chat.new_turn("system")
chat.add_text(SYSTEM_PROMPT)
chat.end_turn()
chat.new_turn("user")
chat.add_audio(wav, sr)
chat.end_turn()
chat.new_turn("assistant")

for t in model.generate_sequential(**chat, max_new_tokens=256):
    if t.numel() == 1:
        print(processor.text.decode(t), end="", flush=True)

## Aufs Gerät

Audio-GGUFs brauchen eigene Runner (`llama-liquid-audio-cli` / `-server`) und
drei zusätzliche Dateien aus dem `-GGUF`-Repo: `mmproj-`, `vocoder-` und
`tokenizer-`. Details: <https://docs.liquid.ai/lfm/models/lfm25-audio-1.5b>